In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# LBCO — neutron powder, constant wavelength, pseudo-Voigt

In [2]:
import easydiffraction as ed
from easydiffraction import ExperimentFactory
from easydiffraction import StructureFactory
from easydiffraction.analysis import verification as verify

## Build the project

In [3]:
project = ed.Project()

## Define the structure

In [4]:
structure = StructureFactory.from_scratch(name='lbco')

structure.space_group.name_h_m = 'P m -3 m'  # FullProf Space group symbol

structure.cell.length_a = 3.890790  # FullProf a

structure.atom_sites.create(
    label='La',  # FullProf Atom
    type_symbol='La',  # FullProf Typ
    fract_x=0.0,  # FullProf X
    fract_y=0.0,  # FullProf Y
    fract_z=0.0,  # FullProf Z
    occupancy=0.5,  # FullProf Occ
    adp_type='Biso',  # FullProf Biso
    adp_iso=0.57511,  # FullProf Biso
)
structure.atom_sites.create(
    label='Ba',  # FullProf Atom
    type_symbol='Ba',  # FullProf Typ
    fract_x=0.0,  # FullProf X
    fract_y=0.0,  # FullProf Y
    fract_z=0.0,  # FullProf Z
    occupancy=0.5,  # FullProf Occ
    adp_type='Biso',  # FullProf Biso
    adp_iso=0.57511,  # FullProf Biso
)
structure.atom_sites.create(
    label='Co',  # FullProf Atom
    type_symbol='Co',  # FullProf Typ
    fract_x=0.5,  # FullProf X
    fract_y=0.5,  # FullProf Y
    fract_z=0.5,  # FullProf Z
    occupancy=1.0,  # FullProf Occ
    adp_type='Biso',  # FullProf Biso
    adp_iso=0.26023,  # FullProf Biso
)
structure.atom_sites.create(
    label='O',  # FullProf Atom
    type_symbol='O',  # FullProf Typ
    fract_x=0.0,  # FullProf X
    fract_y=0.5,  # FullProf Y
    fract_z=0.5,  # FullProf Z
    occupancy=0.97856,  # FullProf Occ
    adp_type='Biso',  # FullProf Biso
    adp_iso=1.36662,  # FullProf Biso
)

project.structures.add(structure)

## Load the FullProf reference

In [5]:
FULLPROF_PROJECT_DIR = 'pd-neut-cwl_pv_lbco'
FULLPROF_PRF_FILE = 'lbco.prf'
FULLPROF_BAC_FILE = 'lbco.bac'
FULLPROF_ZERO = 0.62040  # FullProf Zero
FULLPROF_SCALE = 9.405870  # FullProf Scale
FULLPROF_WAVELENGTH = 1.494000  # FullProf Lambda
FULLPROF_U = 0.081547  # FullProf U
FULLPROF_V = -0.115345  # FullProf V
FULLPROF_W = 0.121125  # FullProf W
FULLPROF_X = 0.0  # FullProf X
FULLPROF_Y = 0.083038  # FullProf Y

x, calc_fullprof = verify.load_fullprof_calc_profile(
    FULLPROF_PROJECT_DIR,
    FULLPROF_PRF_FILE,
    FULLPROF_BAC_FILE,
    FULLPROF_ZERO,
)

## Create the experiment

In [6]:
experiment = ExperimentFactory.from_scratch(
    name='lbco',
    sample_form='powder',
    beam_mode='constant wavelength',
    radiation_probe='neutron',
    scattering_type='bragg',
)
verify.set_reference_as_measured(experiment, x, calc_fullprof)

experiment.linked_phases.create(id='lbco', scale=FULLPROF_SCALE)

experiment.instrument.setup_wavelength = FULLPROF_WAVELENGTH
experiment.instrument.calib_twotheta_offset = FULLPROF_ZERO

experiment.peak.type = 'pseudo-voigt'
experiment.peak.broad_gauss_u = FULLPROF_U
experiment.peak.broad_gauss_v = FULLPROF_V
experiment.peak.broad_gauss_w = FULLPROF_W
experiment.peak.broad_lorentz_x = FULLPROF_X
experiment.peak.broad_lorentz_y = FULLPROF_Y

project.experiments.add(experiment)

Peak profile type for experiment 'lbco' changed to


pseudo-voigt


## ed-cryspy VS FullProf

In [7]:
experiment.calculator.type = 'cryspy'

project.analysis.calculate()
calc_ed_cryspy = experiment.data.intensity_calc

project.display.pattern_comparison(
    'lbco',
    reference=calc_fullprof,
    candidate=calc_ed_cryspy,
    reference_label='FullProf',
    candidate_label='ed-cryspy',
)

Calculator for experiment 'lbco' already set to


cryspy


## ed-crysfml VS FullProf

In [8]:
experiment.calculator.type = 'crysfml'

project.analysis.calculate()
calc_ed_crysfml = experiment.data.intensity_calc

project.display.pattern_comparison(
    'lbco',
    reference=calc_fullprof,
    candidate=calc_ed_crysfml,
    reference_label='FullProf',
    candidate_label='ed-crysfml',
)

Calculator for experiment 'lbco' changed to


crysfml


## Agreement check

In [9]:
verify.assert_patterns_agree(
    [
        ('cryspy vs FullProf', calc_fullprof, calc_ed_cryspy),
        ('crysfml vs FullProf', calc_fullprof, calc_ed_crysfml),
    ],
)

,Comparison,Metric,Expected,Actual,OK
1,cryspy vs FullProf,Profile diff (%),< 2.5,0.68,✅
2,,Max deviation (%),< 6,0.42,✅
3,,Area ratio,0.99 to 1.01,0.9959,✅
4,,Shape correlation,> 0.999,1.0000,✅
5,crysfml vs FullProf,Profile diff (%),< 2.5,0.55,✅
6,,Max deviation (%),< 6,0.34,✅
7,,Area ratio,0.99 to 1.01,0.9913,✅
8,,Shape correlation,> 0.999,1.0000,✅


True